# Data Preparation
This version of the data preparation is before having access to the full text of the articles. Instead, we're running a test using just article titles.

In [1]:
import pandas as pd
import numpy as np

In [2]:
raw_df = pd.read_csv('Data/posts-export-by-page-views-Feb-01-2025-Mar-05-2026-Masthead-Maine.csv')
raw_df.head()

,Apikey,URL,Title,Publish date,Authors,Section,Tags,Sort (Views),Visitors,Views,...,Channel vis.,Website views,AMP views,Fb instant views,Post id,Views source,Views syndicated,Views by Site,High-Level Smart Tags,Low-Level Smart Tags
0,pressherald.com,https://www.pressherald.com,The Portland Press Herald,NaN,Staff,Uncategorized,NaN,13604769,1188026,13604769,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,centralmaine.com,https://www.centralmaine.com,Kennebec Journal and Morning Sentinel,NaN,Staff,Uncategorized,NaN,4571551,453052,4571551,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Law, Gov’t & Politics,Legal Issues",NaN
2,pressherald.com,http://www.pressherald.com/obituaries/,Obituaries,NaN,Staff,Uncategorized,NaN,3465736,410059,3465736,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sunjournal.com,https://www.sunjournal.com,Lewiston Sun Journal,NaN,Staff,Uncategorized,NaN,3436858,367361,3436858,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,centralmaine.com,https://www.centralmaine.com/obituaries/,Obituaries,NaN,Staff,Uncategorized,NaN,3284383,275492,3284383,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
raw_df.shape

(10000, 47)

In [4]:
raw_df.columns

Index(['Apikey', 'URL', 'Title', 'Publish date', 'Authors', 'Section', 'Tags',
       'Sort (Views)', 'Visitors', 'Views', 'Avg. views', 'Engaged minutes',
       'Avg. minutes', 'New vis.', 'Views new vis.', 'Avg. views new vis.',
       'Minutes New Vis.', 'Avg. minutes new vis.', 'Returning vis.',
       'Views ret. vis.', 'Avg. views ret. vis.', 'Minutes Ret. Vis.',
       'Avg. minutes ret. vis.', 'Desktop views', 'Mobile views',
       'Tablet views', 'Search refs', 'Internal refs', 'Other refs',
       'Direct refs', 'Social refs', 'Fb refs', 'Tw refs', 'Pi refs',
       'Social interactions', 'Fb interactions', 'Pi interactions',
       'Channel vis.', 'Website views', 'AMP views', 'Fb instant views',
       'Post id', 'Views source', 'Views syndicated', 'Views by Site',
       'High-Level Smart Tags', 'Low-Level Smart Tags'],
      dtype='str')

In [5]:
raw_df['Post id'].isna().sum()

np.int64(883)

In [6]:
raw_df['Publish date'].isna().sum()

np.int64(883)

In [7]:
i = 500
print(raw_df.URL[i],'\n',raw_df['Post id'][i])

https://www.pressherald.com/2025/08/11/mother-accused-of-killing-3-year-old-in-milford-appears-in-court/ 
 https://www.pressherald.com/2025/08/11/mother-accused-of-killing-3-year-old-in-milford-appears-in-court/


I can either use 'URL' or 'Post id' as identifiable index. Can filter by either 'Post id' or 'Publish date'

In [8]:
art_df = raw_df[~raw_df['Post id'].isna()].reset_index() # full "raw" dataset that only contains articles
art_df.shape

(9117, 48)

## 0. 'User_Needs' column

In [9]:
art_df['Tag_list'] = art_df.Tags.apply(lambda x : x.split(','))

user_tags = []
for i in range(len(art_df)):
    flag = 0
    for tag in art_df['Tag_list'][i]:
        if 'user_need' in tag:
            user_tags.append(tag[len('user_need: '):])
            flag = 1
    if flag:
        continue
    user_tags.append('none') # this is how they're labeled originally

In [10]:
len(user_tags) # just checking that the len matches 

9117

In [11]:
art_df['User_Needs'] = user_tags
art_df.User_Needs.unique()

<StringArray>
[          'update-me',                'none', 'give-me-perspective',
          'educate-me',          'connect-me',             'help-me',
          'inspire-me',      'other-not-news']
Length: 8, dtype: str

NOTE: 'other-not-news' tag is NOT part of the User Needs tags. **Should it be treated as 'none'?**

## 1. Clean data for EDA

(copy-pasted the list of columns for reference)<br />
       'Apikey', 'URL', 'Title', 'Publish date', 'Authors', 'Section', 'Tags',<br />
       'Sort (Views)', 'Visitors', 'Views', 'Avg. views', 'Engaged minutes',<br />
       'Avg. minutes', 'New vis.', 'Views new vis.', 'Avg. views new vis.',<br />
       'Minutes New Vis.', 'Avg. minutes new vis.', 'Returning vis.',<br />
       'Views ret. vis.', 'Avg. views ret. vis.', 'Minutes Ret. Vis.',<br />
       'Avg. minutes ret. vis.', 'Desktop views', 'Mobile views',<br />
       'Tablet views', 'Search refs', 'Internal refs', 'Other refs',<br />
       'Direct refs', 'Social refs', 'Fb refs', 'Tw refs', 'Pi refs',<br />
       'Social interactions', 'Fb interactions', 'Pi interactions',<br />
       'Channel vis.', 'Website views', 'AMP views', 'Fb instant views',<br />
       'Post id', 'Views source', 'Views syndicated', 'Views by Site'<br />

In [12]:
# choose columns to keep
columns = ['Apikey','URL','Title','Publish date','Authors','Section','User_Needs', # metadata
           'Views','Avg. views','Engaged minutes','Avg. minutes', # some metrics (can be changed to include other metrics!)
           'Desktop views','Mobile views', 'Tablet views'] # device metrics
eda_df = art_df[columns]
eda_df.head()

,Apikey,URL,Title,Publish date,Authors,Section,User_Needs,Views,Avg. views,Engaged minutes,Avg. minutes,Desktop views,Mobile views,Tablet views
0,"pressherald.com, sunjournal.com",http://www.pressherald.com/2026/01/09/gray-inv...,Gray investigated for buying $1.25M fire truck...,2026-01-09 08:55,Rory Sweeting,News,update-me,109875,1.084,31960.0,0.315,3910.0,104307.0,1658.0
1,"centralmaine.com, pressherald.com, sunjournal.com",https://www.pressherald.com/2025/03/06/social-...,Social Security now requires Maine parents to ...,2025-03-06 17:14,Joe Lawlor,News,none,98329,1.109,64495.0,0.727,9783.0,86241.0,2305.0
2,"centralmaine.com, pressherald.com, sunjournal.com",http://www.pressherald.com/2026/01/28/ice-agen...,"ICE agents shatter window, leave 1-month-old b...",2026-01-28 17:43,"Dylan Tusinski,Salomé Cloteaux",News,give-me-perspective,76168,1.145,84939.0,1.277,12634.0,61819.0,1715.0
3,"centralmaine.com, pressherald.com, sunjournal.com",https://www.pressherald.com/2025/02/05/maine-m...,Maine Mall shooting: Police search for suspect...,2025-02-05 16:21,"Daniel Kool,Morgan Womack",News,none,73901,1.341,38065.0,0.691,14806.0,58483.0,612.0
4,"centralmaine.com, pressherald.com, sunjournal.com",https://www.pressherald.com/2025/09/01/bob-dyl...,Bob Dylan and Willie Nelson headline Outlaw Mu...,2025-09-01 04:00,Aimsel Ponti,Life & Culture,none,64763,1.155,23763.0,0.424,2635.0,59514.0,2614.0


In [13]:
eda_df.describe()

,Views,Avg. views,Engaged minutes,Avg. minutes,Desktop views,Mobile views,Tablet views
count,9117.000000,9117.000000,9117.000000,9117.000000,9117.000000,9117.000000,9101.000000
mean,3207.525941,1.249237,2093.950093,0.784424,1082.512998,2061.039158,64.085815
std,4675.475642,2.015034,3388.032485,0.362827,1173.264999,3668.032484,110.368077
min,741.000000,0.992000,14.000000,0.009000,4.000000,25.000000,1.000000
25%,1109.000000,1.147000,624.000000,0.526000,458.000000,613.000000,18.000000
50%,1802.000000,1.201000,1104.000000,0.722000,715.000000,1026.000000,33.000000
75%,3434.000000,1.267000,2240.000000,0.982000,1262.000000,2038.000000,67.000000
max,109875.000000,193.000000,84939.000000,11.200000,19047.000000,104307.000000,2614.000000


In [14]:
eda_df['Tablet views'] = eda_df['Tablet views'].fillna(0)
eda_df.describe()

,Views,Avg. views,Engaged minutes,Avg. minutes,Desktop views,Mobile views,Tablet views
count,9117.000000,9117.000000,9117.000000,9117.000000,9117.000000,9117.000000,9117.000000
mean,3207.525941,1.249237,2093.950093,0.784424,1082.512998,2061.039158,63.973346
std,4675.475642,2.015034,3388.032485,0.362827,1173.264999,3668.032484,110.303801
min,741.000000,0.992000,14.000000,0.009000,4.000000,25.000000,0.000000
25%,1109.000000,1.147000,624.000000,0.526000,458.000000,613.000000,18.000000
50%,1802.000000,1.201000,1104.000000,0.722000,715.000000,1026.000000,33.000000
75%,3434.000000,1.267000,2240.000000,0.982000,1262.000000,2038.000000,67.000000
max,109875.000000,193.000000,84939.000000,11.200000,19047.000000,104307.000000,2614.000000


In [15]:
eda_df.count()

Apikey             9117
URL                9117
Title              9117
Publish date       9117
Authors            9117
Section            9117
User_Needs         9117
Views              9117
Avg. views         9117
Engaged minutes    9117
Avg. minutes       9117
Desktop views      9117
Mobile views       9117
Tablet views       9117
dtype: int64

In [16]:
eda_df.to_csv('Data/EDA_data-TITLES.csv', index=False) # save!

## 2. Clean data for ML

In [17]:
# we're just cutting out unneeded columns and splitting into tagged/untagged datasets
valid_tags = ['update-me','give-me-perspective','educate-me',
              'connect-me','help-me','inspire-me']
ml_cols = ['URL','Title','User_Needs']

tagged_df = eda_df[ml_cols][eda_df.User_Needs.isin(valid_tags)]
print(tagged_df.shape[0],' tagged.')

untagged_df = eda_df[ml_cols][~eda_df.User_Needs.isin(valid_tags)]
print(untagged_df.shape[0],' untagged.')

3928  tagged.
5189  untagged.


In [18]:
tagged_df.to_csv('Data/ML_tagged_data-TITLES.csv',index=False)
untagged_df.to_csv('Data/ML_untagged_data-TITLES.csv',index=False)